# ZnO–CO2 site memory: SVM, QSVM and IBM QSVM

How long can the starting adsorption site be identified from the AIMD? For each time
window from the start of the MD, the models predict the starting site of every
trajectory from the CO2 geometry in that window (leave one seed out, site labels
permuted within each seed for the p-value). Each section runs a module exactly as on the
command line, so the notebook and the terminal write the same files:

| Section | Module | Writes |
| --- | --- | --- |
| 1 Frame analysis | `scripts.analysis.md_trajectory_analysis` | `md/<material>/<site>/md_frame_analysis.csv` in each simulation run, tables in `traj_<run>/` |
| 2 Memory curves | `scripts.analysis.site_memory` | `100k/`, `300k/`: windows, predictions, features, summary, curve |
| 3 IBM account | `scripts.models.qml_ibm.ibm_account check` | nothing: login and backend status only |
| 4 IBM plan | `scripts.analysis.site_memory_ibm` (no `--submit`) | `ibm_<T>_bw<λ>/`: plan, SVM and ideal QSVM; IBM is not contacted |
| 5 IBM hardware | `scripts.analysis.site_memory_ibm --submit` | the same folder plus `hardware_kernels/` (raw kernels, job ids, quantum time) |
| 6 Reuse | `scripts.analysis.site_memory_ibm --reuse-hardware` | `ibm_<T>_bw<λ>_reuse/`: the analysis again from the saved kernels, without the QPU |

All outputs go to `results/analyses/site_memory/<OUTPUT_ID>/`. Models: majority,
nearest centroid, logistic, RBF SVM, `qsvm` (standard ZZ feature map, Qiskit
statevector) and `qsvm_bw` (ZZ map on λ·x, λ chosen from the training features only,
without labels, so that the kernel has the width of the RBF kernel). Details:
[docs/analysis.md](../docs/analysis.md), section "Site memory".

Nothing runs until a `RUN_*` flag in the configuration cell is set to `True`. Section 5
is the only one that uses QPU time; read the plan of section 4 first. The IBM sections
need a saved account (section 3).

## Environment

`qml-zno-modeling` (scikit-learn, Qiskit, IBM Runtime). Interactive: `jupyter lab
zno-co2-models.ipynb` from `notebooks/`. Headless:

```bash
cd notebooks
./run_zno_notebook.sh zno-co2-models.ipynb zno-co2-models.executed.ipynb
```

In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import sys

CWD = Path.cwd()
NOTEBOOK_DIR = next(
    (path for path in [CWD, CWD / "notebooks", CWD.parent / "notebooks"] if (path / "zno-co2-models.ipynb").exists()),
    CWD,
)
LUCAS_DIR = next((path for path in [CWD, CWD.parent, NOTEBOOK_DIR.parent]
                  if (path / "scripts" / "analysis" / "site_memory.py").exists()), None)
if LUCAS_DIR is None:
    raise FileNotFoundError("Could not resolve the repository. Run from notebooks/ or the repository root.")
print(f"Repository: {LUCAS_DIR}")

def run_cmd(args: list[str]) -> None:
    """Run a command from the repository root and stream its output into the cell."""
    print("$", " ".join(args), flush=True)
    with subprocess.Popen(args, cwd=LUCAS_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, args)

def md_finished(run: str) -> bool:
    """True when the MD of results/simulations/<run> is marked completed in its run_manifest.json."""
    manifest = LUCAS_DIR / "results" / "simulations" / run / "run_manifest.json"
    if not manifest.is_file():
        return False
    methods = json.loads(manifest.read_text(encoding="utf-8")).get("methods", {})
    return methods.get("md", {}).get("status") == "completed"

def finished_runs(runs: list[str]) -> list[str]:
    """Runs whose MD is complete; the others are reported and left out (a partial run would
    shorten every window to its shortest trajectory)."""
    kept = [run for run in runs if md_finished(run)]
    for run in runs:
        if run not in kept:
            print(f"[skip] {run}: MD not completed (or run missing); left out.")
    return kept

def show_memory(output: Path) -> None:
    """Memory time per model and the windows table as correct/total (p)."""
    summary = output / "site_memory_summary.json"
    windows = output / "site_memory_windows.csv"
    if not summary.is_file() or not windows.is_file():
        print(f"[missing] {output}")
        return
    data = json.loads(summary.read_text(encoding="utf-8"))
    print(f"seeds {data['seeds']}  memory time (fs, p <= {data['alpha']}): {data['memory_time_fs']}")
    with windows.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    table: dict[str, dict[str, str]] = {}
    for row in rows:
        if row["model"] == "majority":
            continue
        cell = f"{row['n_correct']}/{row['n_trajectories']} ({float(row['p_value']):.3f})"
        table.setdefault(row["model"], {})[f"{float(row['t_start_fs']):g}"] = cell
    try:
        import pandas as pd
        with pd.option_context("display.width", 250, "display.max_columns", None):
            print(pd.DataFrame(table).to_string())
    except ImportError:
        for model, cells in table.items():
            print(model, cells)

def show_image(path: Path) -> None:
    if not path.is_file():
        print(f"[missing] {path}")
        return
    import matplotlib.image as mpimg
    import matplotlib.pyplot as plt
    image = mpimg.imread(path)
    fig, ax = plt.subplots(figsize=(8, 8 * image.shape[0] / image.shape[1]))
    ax.imshow(image)
    ax.axis("off")
    plt.show()

def show_csv(path: Path) -> None:
    if not path.is_file():
        print(f"[missing] {path}")
        return
    import pandas as pd
    with pd.option_context("display.width", 250, "display.max_columns", None):
        print(pd.read_csv(path).to_string(index=False))

## Configuration

Edit only this cell. Runs are combined only when they share temperature, time step, log
interval and equilibration, and have distinct seeds (the modules refuse anything else).
Runs whose MD is not completed are left out with a message.

IBM settings. `IBM_WINDOW_STARTS_FS`: window starts chosen before any hardware run, one
where the site is identifiable and one where it is not. `IBM_BANDWIDTH`: `"1"` is the
standard ZZ map (option A, main result) and `"auto"` the matched bandwidth (option B,
sensitivity); each value gets its own folder. With `--scaling fixed` (default) one kernel
per window serves every split: N(N−1)/2 circuits per window (36 for 3 seeds, 153 for 6).

In [ ]:
MATERIAL = "ZnO"
RUNS_100K = ["zno_md_100k", "zno_md_100k_extra"]       # results/simulations/<run>/
RUNS_300K = ["zno_ic_final", "zno_md_300k_extra"]
OUTPUT_ID = "notebook"                                   # results/analyses/site_memory/<OUTPUT_ID>/

IBM_TEMPERATURE = "300k"          # "100k" or "300k": which runs go to the IBM sections
IBM_WINDOW_STARTS_FS = [0, 800]
IBM_BANDWIDTH = "1"               # "1" standard ZZ (A) or "auto" matched bandwidth (B)
IBM_BACKEND = "ibm_marrakesh"     # None = least busy backend of the account
IBM_ACCOUNT = "zno"               # saved account name (section 3); None = default account
IBM_SHOTS = 512

RUN_FRAMES      = False  # 1) frame analysis of every run (needed once per new run)
RUN_MEMORY_100K = False  # 2) memory curve at 100 K
RUN_MEMORY_300K = False  # 2) memory curve at 300 K
RUN_IBM_CHECK   = False  # 3) login and backend status (no circuits, no QPU time)
RUN_IBM_PLAN    = False  # 4) plan, SVM and ideal QSVM; IBM is NOT contacted
RUN_IBM_SUBMIT  = False  # 5) runs the circuits on IBM hardware (uses QPU time)
RUN_IBM_REUSE   = False  # 6) redo section 5 from its saved kernels, without the QPU

SIMULATIONS = LUCAS_DIR / "results" / "simulations"
OUTPUT_DIR = LUCAS_DIR / "results" / "analyses" / "site_memory" / OUTPUT_ID
IBM_RUNS = {"100k": RUNS_100K, "300k": RUNS_300K}[IBM_TEMPERATURE]
IBM_DIR = OUTPUT_DIR / f"ibm_{IBM_TEMPERATURE}_bw{IBM_BANDWIDTH}"

def run_paths(runs: list[str]) -> list[str]:
    return [str(SIMULATIONS / run) for run in runs]

def ibm_args(runs: list[str], output: Path) -> list[str]:
    args = [sys.executable, "-m", "scripts.analysis.site_memory_ibm", "--runs", *run_paths(runs),
            "--material", MATERIAL, "--window-starts-fs", *[f"{t:g}" for t in IBM_WINDOW_STARTS_FS],
            "--qsvm-bandwidth", IBM_BANDWIDTH, "--shots", str(IBM_SHOTS), "--output-dir", str(output)]
    if IBM_BACKEND:
        args += ["--backend", IBM_BACKEND]
    if IBM_ACCOUNT:
        args += ["--account", IBM_ACCOUNT]
    return args

print(f"Outputs: {OUTPUT_DIR}")
for run in RUNS_100K + RUNS_300K:
    print(f"  {run}: " + ("MD completed" if md_finished(run) else "MD not completed or missing"))

## 1) Frame analysis

`md_trajectory_analysis` on every finished run: one row per saved frame with the CO2
geometry (`md_frame_analysis.csv`, written into the simulation run; nothing else in the
run changes) and per-trajectory tables in `traj_<run>/`. Needed once per new run, and
again only if the run changes.

In [ ]:
if RUN_FRAMES:
    for run in finished_runs(RUNS_100K + RUNS_300K):
        run_cmd([sys.executable, "-m", "scripts.analysis.md_trajectory_analysis",
                 "--results-dir", str(SIMULATIONS / run), "--material", MATERIAL,
                 "--output-dir", str(OUTPUT_DIR / f"traj_{run}")])

## 2) Memory curves at 100 K and 300 K

`site_memory` with every model, 40 fs windows and the same C (1) for all. The memory time
of a model is the end of the run of significant windows (p ≤ 0.05) that starts at t = 0.
With 3 seeds the smallest attainable p is 1/36 = 0.028 (one error in 9 already gives
0.056); with 6 seeds it is 1/7776, so compare temperatures with the same number of seeds.
Isolated significant windows late in the run are expected by chance (about 1 in 20 per
model) and do not count.

In [ ]:
for flag, label, runs in ((RUN_MEMORY_100K, "100k", RUNS_100K), (RUN_MEMORY_300K, "300k", RUNS_300K)):
    if not flag:
        continue
    kept = finished_runs(runs)
    if not kept:
        print(f"[skip] {label}: no finished run")
        continue
    output = OUTPUT_DIR / label
    run_cmd([sys.executable, "-m", "scripts.analysis.site_memory", "--runs", *run_paths(kept),
             "--material", MATERIAL, "--output-dir", str(output)])
    show_memory(output)
    show_image(output / "site_memory_curve.png")

## 3) IBM account

Once per machine, in a **terminal** with the modeling environment (the key is asked
hidden there, so it never appears in this notebook or its outputs):

1. Sign in at [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com). The free Open Plan
   gives a few minutes of QPU time per period (see the dashboard).
2. Dashboard → **API key** → **Create**; copy the key (shown once) and keep it private.
3. **Instances** → your instance → copy its CRN (starts with `crn:v1:`).
4. Save the account (it asks for the key and the CRN):

   ```bash
   python -m scripts.models.qml_ibm.ibm_account save --name zno
   ```

   `--overwrite` replaces it later (new or expired key).

Then set `IBM_ACCOUNT` (and `IBM_BACKEND`) in the configuration cell and run the check
below: login, operational backends and the queue of `IBM_BACKEND`. It sends no circuit.
Never paste the key into this notebook. Full guide:
[docs/analysis.md](../docs/analysis.md#ibm-quantum-account-once-per-machine).

In [ ]:
if RUN_IBM_CHECK:
    check = [sys.executable, "-m", "scripts.models.qml_ibm.ibm_account", "check"]
    if IBM_ACCOUNT:
        check += ["--name", IBM_ACCOUNT]
    if IBM_BACKEND:
        check += ["--backend", IBM_BACKEND]
    run_cmd(check)

## 4) IBM plan (IBM is not contacted)

`site_memory_ibm` without `--submit`: writes the plan (windows, circuits, shots) and
evaluates the RBF SVM and the ideal QSVM on the chosen windows, with the same rows,
splits, scaling and C as the hardware run. Check the number of circuits against the QPU
time of your plan before section 5.

In [ ]:
if RUN_IBM_PLAN and (kept := finished_runs(IBM_RUNS)):
    run_cmd(ibm_args(kept, IBM_DIR))
    show_csv(IBM_DIR / "site_memory_ibm_windows.csv")

## 5) IBM hardware

`site_memory_ibm --submit` runs the compute-uncompute circuits on IBM (QPU time) and
saves the raw kernels, job ids and quantum time in `hardware_kernels/`. Needs a saved
Qiskit Runtime account (section 3). The cell refuses a folder
that already holds hardware kernels, so an earlier hardware run is never overwritten;
change `OUTPUT_ID` or `IBM_BANDWIDTH` for a new one.

In [ ]:
if RUN_IBM_SUBMIT:
    if (IBM_DIR / "hardware_kernels").exists():
        raise SystemExit(f"{IBM_DIR / 'hardware_kernels'} exists: change OUTPUT_ID or IBM_BANDWIDTH.")
    if kept := finished_runs(IBM_RUNS):
        run_cmd([*ibm_args(kept, IBM_DIR), "--submit"])
        show_csv(IBM_DIR / "site_memory_ibm_windows.csv")
        show_csv(IBM_DIR / "site_memory_ibm_kernel_deviation.csv")

## 6) Reuse the hardware kernels

The analysis of section 5 again from its saved kernels, without the QPU (for example
after changing the permutation count). The module refuses kernels computed on other data
(runs, features, scaling or bandwidth).

In [ ]:
if RUN_IBM_REUSE and (kept := finished_runs(IBM_RUNS)):
    reuse_dir = IBM_DIR.with_name(IBM_DIR.name + "_reuse")
    run_cmd([*ibm_args(kept, reuse_dir), "--reuse-hardware", str(IBM_DIR / "hardware_kernels")])
    show_csv(reuse_dir / "site_memory_ibm_windows.csv")